In [ ]:
import pandas as pd
from scipy import integrate

# --- config: where the pipeline wrote its output, and which files to integrate ---
OUTDIR = "/scratch/lappi/pagimeno/incl/proton/files"   # dir containing D0_incl_*.dat
FRAG_TAG = "KniehlKramer"                               # or "BCFY", depending on the build
CHANNEL_TAG = "An0n"
y_vals = [0.0, 1.0, 2.0]

# D0_incl_<frag>_<channel>_Pb_y<Y>.dat holds ALL b and pT for one y, as
# whitespace-separated columns "b  pD0  dsigma_dy" (see run_many_Pb.sh).
# b still needs to be integrated over here (Simpson's rule, weighted by b,
# the radial measure in 2D polar coordinates) -- that's what this cell does.

def integrate_over_b(y):
    ytag = f"{y:.1f}".replace(".", "")
    path = f"{OUTDIR}/D0_incl_{FRAG_TAG}_{CHANNEL_TAG}_Pb_y{ytag}.dat"

    df = pd.read_csv(
        path, comment="#", delim_whitespace=True,
        names=["b", "pt", "result"],
    )

    # rows = pT, columns = b, values = dsigma_dy
    table = df.pivot(index="pt", columns="b", values="result").sort_index(axis=1)
    b_values = table.columns.to_numpy()

    integrated_values = [
        integrate.simpson(row.to_numpy() * b_values, b_values)
        for _, row in table.iterrows()
    ]

    res = pd.DataFrame({"pt": table.index, "result": integrated_values})
    res.to_csv(f"y{ytag}_result.csv", index=False)
    return res

results = {y: integrate_over_b(y) for y in y_vals}
results[y_vals[0]]
